# Ethics & Fairness Diagnostics

Development-only subgroup diagnostics across the canonical gender groups and age bands `22-29`, `30-39`, `40-49`, `50-60`. These are aggregate academic diagnostics, not evidence that the system is suitable for employment decisions.

## Subgroup performance and top-20% representation

In [1]:
from pathlib import Path
import json, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve, f1_score

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "analysis_registry.yaml").exists():
    for parent in ROOT.parents:
        if (parent / "configs" / "analysis_registry.yaml").exists():
            ROOT = parent
            break
registry = yaml.safe_load((ROOT / "configs" / "analysis_registry.yaml").read_text())
frames = {}
for model, meta in registry["models"].items():
    d = pd.read_csv(ROOT / meta["oof_path"]).sort_values("Employee_ID").reset_index(drop=True)
    frames[model] = d
print(f"Loaded {len(frames)} headline-model OOF files from development data only.")
from src.evaluation.fairness import subgroup_classification_metrics, top_k_representation
raw=pd.read_csv(ROOT/"data/raw/Extended_Employee_Performance_and_Productivity_Data.csv")
reg=pd.read_csv(ROOT/"data/processed/split_registry.csv")
dev=raw.merge(reg[["Employee_ID","partition"]],on="Employee_ID"); dev=dev[dev.partition=="development"].copy()
dev["Age_Band"]=pd.cut(dev.Age,bins=[21,29,39,49,60],labels=["22-29","30-39","40-49","50-60"],include_lowest=True)
out=ROOT/"results/fairness"; out.mkdir(parents=True,exist_ok=True)
sub=[]; top=[]
for model,d in frames.items():
    z=d[["Employee_ID","y_true","y_probability"]].merge(dev[["Employee_ID","Gender","Age_Band"]],on="Employee_ID",validate="one_to_one")
    for group_col in ["Gender","Age_Band"]:
        s=subgroup_classification_metrics(z,group_col,threshold=.5,min_count=30); s.insert(0,"model",model); s.insert(1,"grouping",group_col); s=s.rename(columns={group_col:"group"}); sub.append(s)
        k=top_k_representation(z,group_col,.20); k.insert(0,"model",model); k.insert(1,"grouping",group_col); k=k.rename(columns={group_col:"group"}); top.append(k)
subgroups=pd.concat(sub,ignore_index=True); representation=pd.concat(top,ignore_index=True)
subgroups.to_csv(out/"subgroup_metrics.csv",index=False); representation.to_csv(out/"top20_representation.csv",index=False)
summary={"age_bands":["22-29","30-39","40-49","50-60"],"scope":"development OOF only","prohibited_use":"No real hiring, firing, promotion, surveillance, disciplinary action, or automated retention decision.","interpretation":"Near-no-skill predictive performance means these subgroup results are descriptive and do not establish operational fairness."}; (out/"fairness_summary.json").write_text(json.dumps(summary,indent=2))
display(subgroups.round(5)); display(representation.round(5)); summary

Loaded 6 headline-model OOF files from development data only.


/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group,g in df.groupby(group_col,dropna=False):
/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group,g in df.groupby(group_col,dropna=False):
/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or

/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group,g in df.groupby(group_col,dropna=False):
/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group,g in df.groupby(group_col,dropna=False):


/mnt/data/erp_upgrade/Employee_Resignation_Prediction/src/evaluation/fairness.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group,g in df.groupby(group_col,dropna=False):


,model,grouping,group,n,prevalence,average_precision,roc_auc,precision,recall,false_positive_rate,predicted_positive_rate,stable_rate_estimates
0,FT-Transformer,Gender,Female,38303,0.10054,0.09981,0.49756,0.10069,0.55103,0.55013,0.55022,True
1,FT-Transformer,Gender,Male,38537,0.09990,0.10002,0.49527,0.09857,0.48987,0.49722,0.49648,True
2,FT-Transformer,Gender,Other,3160,0.09715,0.10694,0.50915,0.10085,0.46254,0.44374,0.44557,True
3,FT-Transformer,Age_Band,22-29,16391,0.10085,0.09573,0.48846,0.09887,0.51180,0.52321,0.52205,True
4,FT-Transformer,Age_Band,30-39,20368,0.09873,0.09771,0.49027,0.09622,0.49677,0.51114,0.50972,True
5,FT-Transformer,Age_Band,40-49,20605,0.09993,0.10562,0.51096,0.10268,0.53861,0.52259,0.52419,True
6,FT-Transformer,Age_Band,50-60,22636,0.10095,0.10130,0.49741,0.10071,0.52341,0.52479,0.52465,True
7,CatBoost,Gender,Female,38303,0.10054,0.09876,0.49685,0.09917,0.25266,0.25653,0.25614,True
8,CatBoost,Gender,Male,38537,0.09990,0.10100,0.50504,0.10004,0.25896,0.25857,0.25861,True
9,CatBoost,Gender,Other,3160,0.09715,0.09125,0.48104,0.07730,0.20521,0.26358,0.25791,True


,model,grouping,group,population_count,selected_count,population_share,selected_share,representation_ratio
0,FT-Transformer,Gender,Male,38537,6865,0.48171,0.42906,0.89070
1,FT-Transformer,Gender,Female,38303,8687,0.47879,0.54294,1.13398
2,FT-Transformer,Gender,Other,3160,448,0.03950,0.02800,0.70886
3,FT-Transformer,Age_Band,50-60,22636,4614,0.28295,0.28838,1.01917
4,FT-Transformer,Age_Band,40-49,20605,4139,0.25756,0.25869,1.00437
5,FT-Transformer,Age_Band,30-39,20368,3827,0.25460,0.23919,0.93946
6,FT-Transformer,Age_Band,22-29,16391,3420,0.20489,0.21375,1.04326
7,CatBoost,Gender,Male,38537,7749,0.48171,0.48431,1.00540
8,CatBoost,Gender,Female,38303,7605,0.47879,0.47531,0.99274
9,CatBoost,Gender,Other,3160,646,0.03950,0.04038,1.02215


{'age_bands': ['22-29', '30-39', '40-49', '50-60'],
 'scope': 'development OOF only',
 'prohibited_use': 'No real hiring, firing, promotion, surveillance, disciplinary action, or automated retention decision.',
 'interpretation': 'Near-no-skill predictive performance means these subgroup results are descriptive and do not establish operational fairness.'}

## Limitations and mitigation

Sensitive attributes can produce direct or proxy discrimination. The weak predictive signal also makes threshold decisions unreliable. The project therefore prohibits real employment use, reports subgroup denominators and rates, and treats all model explanations as associations rather than causes.